Classifying MNIST Digits with CNNs and Domain Adaptation
---
<p align="center">
<img img src="https://github.com/AleksCipri/LSST_DSFP_DA/raw/main/major_challenges.png" alt="fine" width="400"/>
</p>

This tutorial mirrors the structure of LSST_Tutorial_Mergers which you'll be working on in this hands-on activity.

Here we will use **MNIST** (clean grayscale digits) as the source domain $\mathcal{D}_s$ and **MNIST-M** (same digits on colorful textured backgrounds) as the target domain $\mathcal{D}_t$ — a standard domain adaptation benchmark.

We classify three digits — **1, 4, and 8** — and apply two domain adaptation (DA) techniques to close the performance gap:

- **Baseline**: train on source only, evaluate on both
- **MMD**: Maximum Mean Discrepancy alignment of latent representations [[Paper]](https://www.jmlr.org/papers/volume13/gretton12a/gretton12a.pdf)
- **DANN**: Domain-Adversarial Neural Network with gradient reversal [[Paper]](https://arxiv.org/pdf/1505.07818)

For each model we report: training curves, confusion matrices, ROC curves (one-vs-rest), and a 2D t-SNE of the latent space for both domains.

##### Author: [Aleksandra Ciprijanovic](https://alexciprijanovic.com)

## Problem Setup

**Notation:**
- Source domain: $\mathcal{D}_s = \{(x_s^{(i)}, y_s^{(i)})\}_{i=1}^{N_s}$ — labelled MNIST images, $y_s^{(i)} \in \{0, 1, 2\}$ (mapped from digits 1, 4, 8)
- Target domain: $\mathcal{D}_t = \{x_t^{(j)}\}_{j=1}^{N_t}$ — **unlabelled** MNIST-M images (labels exist but are never used during DA training)
- We assume $p_s(y|x) = p_t(y|x)$ (same digit structure) but $p_s(x) \neq p_t(x)$ (texturized backgrounds)

**Neural network decomposition:**  
$f_\theta = g_\theta \circ \phi_\theta$, where $\phi_\theta$ is the feature extractor and $g_\theta$ is the classifier head.  
The **latent vector** $z = \phi_\theta(x) \in \mathbb{R}^{64}$ is what DA methods operate on.

**Classification loss (cross-entropy, 3 classes):**
$$\mathcal{L}_{\text{CE}}(\theta) = -\frac{1}{N_s}\sum_{i=1}^{N_s} \log f_\theta^{(y_s^{(i)})}(x_s^{(i)})$$

**Before you start:** make a prediction — how large do you expect the source-to-target accuracy gap to be for the baseline model? Remember that the digits themselves are the same; only the backgrounds differ. Does this make the shift easier or harder than a noise-based shift?

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                              classification_report, roc_auc_score)
from sklearn.preprocessing import StandardScaler, label_binarize
import os, random, copy, zipfile, urllib.request
from tqdm import tqdm
from typing import Optional, List

import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import MNIST, ImageFolder
from torch.autograd import Function

def set_all_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_all_seeds()

## 1. Data

We use two datasets:
- **MNIST** (source domain $\mathcal{D}_s$): 28×28 grayscale hand-written digits, replicated to 3 channels to match MNIST-M
- **MNIST-M** (target domain $\mathcal{D}_t$): same digits placed on colorful textured backgrounds — a standard covariate shift benchmark

We focus on **digits 1, 4, and 8**. MNIST is built into `torchvision`; MNIST-M is downloaded from GitHub.

Because the original MNIST labels are `{1, 4, 8}`, we remap them to `{0, 1, 2}` for PyTorch compatibility using the `MappedDataset` wrapper below. This keeps the downstream training code label-index–agnostic.

In [ ]:
TARGET_DIGITS = [1, 4, 8]
label_map     = {1: 0, 4: 1, 8: 2}
CLASS_NAMES   = {0: 'Digit 1', 1: 'Digit 4', 2: 'Digit 8'}

mnist_mean = (0.1307,) * 3
mnist_std  = (0.3015,) * 3

class MappedDataset(Dataset):
    """Wraps a dataset and remaps integer labels via label_map."""
    def __init__(self, dataset, label_map):
        self.dataset = dataset
        self.label_map = label_map
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return img, torch.tensor(self.label_map[int(label)], dtype=torch.long)

mnist_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mnist_mean, mnist_std),
])

print('Loading MNIST...')
mnist_train_full = MNIST(root='../data', train=True,  download=True, transform=mnist_transform)
mnist_test_full  = MNIST(root='../data', train=False, download=True, transform=mnist_transform)

def filter_digits(dataset, allowed):
    return [i for i, (_, y) in enumerate(dataset) if int(y) in allowed]

src_train_idx = filter_digits(mnist_train_full, TARGET_DIGITS)
src_test_idx  = filter_digits(mnist_test_full,  TARGET_DIGITS)
print(f'MNIST filtered  train={len(src_train_idx):,}  test={len(src_test_idx):,}')

In [ ]:
mnistm_mean = (0.4579, 0.4621, 0.4082)
mnistm_std  = (0.1880, 0.1755, 0.1956)

mnistm_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize(mnistm_mean, mnistm_std),
])

zip_path     = '../data/MNIST-M.zip'
extract_path = '../data/mnist_m'

if not os.path.exists(zip_path):
    print('Downloading MNIST-M...')
    urllib.request.urlretrieve(
        'https://github.com/mashaan14/MNIST-M/raw/main/MNIST-M.zip', zip_path)

if not os.path.exists(extract_path):
    print('Extracting MNIST-M...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_path)

mnistm_train_full = ImageFolder(os.path.join(extract_path, 'MNIST-M/training'), transform=mnistm_transform)
mnistm_test_full  = ImageFolder(os.path.join(extract_path, 'MNIST-M/testing'),  transform=mnistm_transform)

tgt_train_idx = filter_digits(mnistm_train_full, TARGET_DIGITS)
tgt_test_idx  = filter_digits(mnistm_test_full,  TARGET_DIGITS)
print(f'MNIST-M filtered  train={len(tgt_train_idx):,}  test={len(tgt_test_idx):,}')

In [ ]:
def unnormalize(img, mean, std):
    img = img.clone()
    for c in range(img.shape[0]):
        img[c] = img[c] * std[c] + mean[c]
    return img

def collect_one_per_class(full_ds, indices, n_classes):
    ds = MappedDataset(Subset(full_ds, indices), label_map)
    found = {}
    for img, label in ds:
        c = int(label)
        if c not in found: found[c] = img
        if len(found) == n_classes: break
    return found

src_ex = collect_one_per_class(mnist_train_full,  src_train_idx, 3)
tgt_ex = collect_one_per_class(mnistm_train_full, tgt_train_idx, 3)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
fig.suptitle('Source (MNIST) vs Target (MNIST-M)', fontsize=13)
for j in range(3):
    axes[0, j].imshow(unnormalize(src_ex[j], mnist_mean,   mnist_std).permute(1,2,0).numpy())
    axes[0, j].set_title(CLASS_NAMES[j], fontsize=10); axes[0, j].axis('off')
    axes[1, j].imshow(unnormalize(tgt_ex[j], mnistm_mean, mnistm_std).permute(1,2,0).numpy())
    axes[1, j].set_title(CLASS_NAMES[j], fontsize=10); axes[1, j].axis('off')
axes[0, 0].set_ylabel('MNIST', fontsize=11)
axes[1, 0].set_ylabel('MNIST-M', fontsize=11)
plt.tight_layout(); plt.show()

### Data Splits and DataLoaders

We take random subsets of 5,000 training / 1,000 validation / 1,000 test samples from MNIST and MNIST-M. The source validation set is drawn from MNIST training data and is used to track source overfitting. Target labels are available here but are **never used during DA training** (only source and target images are processed to calculate DA loss, while source labels are used for CE loss).

> **Why do we normalise source and target separately?** Think about what each domain's pixel distribution looks like before answering.

In [ ]:
RS = 42
BATCH = 64
N_TRAIN, N_VAL, N_TEST = 5000, 1000, 1000

rng = np.random.default_rng(RS)

def sample_n(indices, n, rng):
    chosen = rng.choice(len(indices), size=min(n, len(indices)), replace=False)
    return [indices[i] for i in chosen]

# Source: MNIST — train / val / test
src_pool    = sample_n(src_train_idx, N_TRAIN + N_VAL, rng)
src_train_i, src_val_i = src_pool[:N_TRAIN], src_pool[N_TRAIN:]
src_test_i  = sample_n(src_test_idx, N_TEST, rng)

# Target: MNIST-M — train / val / test
tgt_pool    = sample_n(tgt_train_idx, N_TRAIN + N_VAL, rng)
tgt_train_i, tgt_val_i = tgt_pool[:N_TRAIN], tgt_pool[N_TRAIN:]
tgt_test_i  = sample_n(tgt_test_idx, N_TEST, rng)

def make_loader(full_ds, indices, shuffle):
    ds = MappedDataset(Subset(full_ds, indices), label_map)
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

source_train_loader = make_loader(mnist_train_full,   src_train_i, shuffle=True)
source_val_loader   = make_loader(mnist_train_full,   src_val_i,   shuffle=False)
source_test_loader  = make_loader(mnist_test_full,    src_test_i,  shuffle=False)
target_train_loader = make_loader(mnistm_train_full, tgt_train_i, shuffle=True)
target_val_loader   = make_loader(mnistm_train_full, tgt_val_i,   shuffle=False)
target_test_loader  = make_loader(mnistm_test_full,  tgt_test_i,  shuffle=False)

imgs, labs = next(iter(source_train_loader))
print(f'Source batch: {imgs.shape}  dtype={imgs.dtype}  labels={labs[:8].tolist()}')
imgs, labs = next(iter(target_train_loader))
print(f'Target batch: {imgs.shape}  dtype={imgs.dtype}  labels={labs[:8].tolist()}')
print(f'\nSource  train={N_TRAIN}  val={N_VAL}  test={N_TEST}')
print(f'Target  train={N_TRAIN}  val={N_VAL}  test={N_TEST}')

## 2. Model Architecture

We use a CNN with three convolutional blocks followed by two fully connected layers:

| Block | Layer | Filters / Units | Kernel | Other |
|-------|-------|-----------------|--------|-------|
| 1 | Conv2d | 8 | 5×5 | BatchNorm, ReLU, MaxPool(2), Dropout(0.1) |
| 2 | Conv2d | 16 | 3×3 | BatchNorm, ReLU, MaxPool(2), Dropout(0.1) |
| 3 | Conv2d | 32 | 3×3 | BatchNorm, ReLU, MaxPool(2), Dropout(0.2) |
| FC1 | Linear | 64 | — | ReLU — **this output is the latent vector $z$** |
| FC2 | Linear | 32 | — | ReLU |
| Out | Linear | 3 | — | (logits) |

After three MaxPool(2) layers: 28×28 → 14×14 → 7×7 → 3×3, so the flattened dimension is 32×3×3 = 288.

**Questions to consider before writing the forward pass:**
- Why does the model need to return *both* logits *and* the latent vector $z$? When would you need one vs. the other?
- The latent vector $z$ is 64-dimensional. What are the trade-offs of making it larger or smaller for DA? Can alignment be performed on some other layer?
- Remember, we are classifying 3 out of 10 digits.

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 8, 5, padding=2), nn.BatchNorm2d(8),
            nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout2d(0.1),
        )  # 28 -> 14
        self.block2 = nn.Sequential(
            nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16),
            nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout2d(0.1),
        )  # 14 -> 7
        self.block3 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2), nn.Dropout2d(0.2),
        )  # 7 -> 3
        self.fc1 = nn.Sequential(nn.Linear(32 * 3 * 3, 64), nn.ReLU(inplace=True))
        self.fc2 = nn.Sequential(nn.Linear(64, 32), nn.ReLU(inplace=True))
        self.out = nn.Linear(32, num_classes)

    def forward(self, x):
        ## TO DO ##
        # 1. Pass x through block1, block2, block3 in sequence.
        # 2. Flatten x to shape (batch_size, 32*3*3) using x.view(x.size(0), -1).
        # 3. Pass through fc1 to obtain the 64-dim latent vector z.
        # 4. Pass z through fc2 and out to obtain logits.
        # 5. Return (logits, z) — both are needed: logits for classification, z for DA alignment.
        pass

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
_x = torch.randn(4, 3, 28, 28).to(device)
_logits, _z = CNN().to(device)(_x)
print(f'Logits: {_logits.shape}   Latent z: {_z.shape}')  # expect [4,3] and [4,64]

## 3. Evaluation Functions

The helper functions below are provided. Read through them to understand what each one returns — you will interpret their outputs throughout the tutorial.

Because we have **3 classes**, ROC curves are computed **one-vs-rest (OvR)** for each class separately using `label_binarize`. A macro-average AUC is reported in the final comparison table.

> **Note:** `get_predictions` returns `y_prob` as a matrix of shape `(N, 3)` — one softmax probability column per class. This is different from the binary case where a single probability suffices.

In [ ]:
def get_predictions(model, loader, device):
    """Returns (y_true, y_pred, y_prob, z_latent).
    y_prob has shape (N, num_classes) for multi-class ROC.
    """
    model.eval()
    y_true, y_pred, y_prob, z_all = [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits, z = model(imgs.to(device))
            probs = torch.softmax(logits, dim=1).cpu()
            preds = logits.argmax(1).cpu()
            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())
            y_prob.append(probs.numpy())
            z_all.append(z.cpu())
    return (np.array(y_true), np.array(y_pred),
            np.concatenate(y_prob, axis=0), torch.cat(z_all, dim=0))


def plot_history(tl, vl, ta, va, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13)
    epochs = range(1, len(tl)+1)
    ax1.plot(epochs, tl, label='Train'); ax1.plot(epochs, vl, label='Val')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('CE Loss'); ax1.set_title('CE Loss'); ax1.legend()
    ax2.plot(epochs, ta, label='Train'); ax2.plot(epochs, va, label='Val')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Accuracy'); ax2.legend()
    plt.tight_layout(); plt.show()


def plot_conf_matrix(y_true, y_pred, title, ax):
    n = len(CLASS_NAMES)
    cm = confusion_matrix(y_true, y_pred, labels=list(CLASS_NAMES.keys()))
    im = ax.imshow(cm, cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ticks = list(CLASS_NAMES.values())
    ax.set_xticks(range(n)); ax.set_xticklabels(ticks, fontsize=9)
    ax.set_yticks(range(n)); ax.set_yticklabels(ticks, fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    thresh = cm.max() / 2
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > thresh else 'black', fontsize=11)


def plot_roc_multiclass(y_true, y_prob, domain_label, ax):
    """OvR ROC for each class; returns macro AUC."""
    y_bin = label_binarize(y_true, classes=list(CLASS_NAMES.keys()))
    cmap = plt.colormaps['tab10']
    aucs = []
    for i, cls_name in CLASS_NAMES.items():
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        ax.plot(fpr, tpr, lw=1.8, color=cmap(i),
                label=f'{cls_name} (AUC={roc_auc:.3f})')
    ax.plot([0,1],[0,1], 'k--', lw=0.8)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves (OvR) — {domain_label}')
    ax.legend(fontsize=9)
    return float(np.mean(aucs))


def plot_tsne(z_src, ys, z_tgt, yt, title, ax, subset=600):
    from sklearn.manifold import TSNE
    z = torch.cat([z_src, z_tgt], dim=0)
    z = torch.tensor(StandardScaler().fit_transform(z.numpy()), dtype=torch.float32)
    domains = np.array(['source']*len(z_src) + ['target']*len(z_tgt))
    y = np.concatenate([ys, yt])
    rng2 = np.random.default_rng(42)
    idx = rng2.choice(len(z), min(subset, len(z)), replace=False)
    z, y, domains = z[idx], y[idx], domains[idx]
    z_2d = TSNE(n_components=2, perplexity=100, random_state=42).fit_transform(z.numpy())
    cmap = plt.colormaps['tab10']
    for cls in CLASS_NAMES.keys():
        for dom, mk in [('source', 'o'), ('target', '^')]:
            mask = (y == cls) & (domains == dom)
            ax.scatter(z_2d[mask, 0], z_2d[mask, 1], c=[cmap(cls)], marker=mk,
                       s=15, alpha=0.7,
                       label=f"{CLASS_NAMES[cls]} ({'src' if dom=='source' else 'tgt'})")
    ax.set_title(title); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')


def full_eval(model, src_loader, tgt_loader, device, name):
    ys_t, ys_p, ys_prob, zs = get_predictions(model, src_loader, device)
    yt_t, yt_p, yt_prob, zt = get_predictions(model, tgt_loader, device)

    print(f'\n=== {name} ===')
    print(f'Source accuracy: {100*(ys_p==ys_t).mean():.2f}%')
    print(f'Target accuracy: {100*(yt_p==yt_t).mean():.2f}%')
    print('\nSource report:'); print(classification_report(ys_t, ys_p, target_names=list(CLASS_NAMES.values())))
    print('Target report:'); print(classification_report(yt_t, yt_p, target_names=list(CLASS_NAMES.values())))

    # Confusion matrices
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{name} — Confusion Matrices', fontsize=12)
    plot_conf_matrix(ys_t, ys_p, 'Source (MNIST) test',   axes[0])
    plot_conf_matrix(yt_t, yt_p, 'Target (MNIST-M) test', axes[1])
    plt.tight_layout(); plt.show()

    # ROC — one subplot per domain
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'{name} — ROC Curves (OvR)', fontsize=12)
    plot_roc_multiclass(ys_t, ys_prob, 'Source (MNIST)',   ax1)
    plot_roc_multiclass(yt_t, yt_prob, 'Target (MNIST-M)', ax2)
    plt.tight_layout(); plt.show()

    # t-SNE
    fig, ax = plt.subplots(figsize=(7, 5))
    plot_tsne(zs, ys_t, zt, yt_t, f'{name} — Latent Space (t-SNE)', ax)
    ax.legend(markerscale=1.5, fontsize=9, loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout(); plt.show()

    return ys_t, ys_p, ys_prob, yt_t, yt_p, yt_prob

## 4. Baseline: Cross-Entropy Only (No Domain Adaptation)

We first train the CNN on MNIST (source) for 20 epochs using only the cross-entropy loss, then evaluate on both source and target test sets.

This establishes the **performance gap** — the drop in accuracy when moving from clean MNIST to textured MNIST-M images. The gap is your target to close with DA.

**Your task:** fill in the two lines inside the training loop that perform the forward pass and compute the loss. Everything else — the val loop, logging, and best-model saving — is provided.

In [ ]:
NUM_EPOCHS = 20
CE_model  = CNN(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CE_model.parameters(), lr=1e-3)

tl_ce, vl_ce, ta_ce, va_ce = [], [], [], []
best_val, best_state_ce = float('inf'), None

for epoch in tqdm(range(NUM_EPOCHS)):
    CE_model.train()
    t_loss = t_correct = t_total = 0

    for imgs, labels in source_train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        ## TO DO ##
        # Forward pass: call CE_model on imgs to get (logits, _).
        # Compute the cross-entropy loss using criterion.

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        t_loss    += loss.item()
        t_correct += logits.argmax(1).eq(labels).sum().item()
        t_total   += labels.size(0)

    CE_model.eval()
    v_loss = v_correct = v_total = 0
    with torch.no_grad():
        for imgs, labels in source_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _  = CE_model(imgs)
            v_loss    += criterion(logits, labels).item()
            v_correct += logits.argmax(1).eq(labels).sum().item()
            v_total   += labels.size(0)

    tl = t_loss / len(source_train_loader)  # train CE loss
    vl = v_loss / len(source_val_loader)    # val CE loss
    ta = 100 * t_correct / t_total
    va = 100 * v_correct / v_total

    tl_ce.append(tl); vl_ce.append(vl)
    ta_ce.append(ta); va_ce.append(va)

    if vl < best_val:
        best_val = vl; best_state_ce = copy.deepcopy(CE_model.state_dict())
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}  CE={tl:.4f}  val_CE={vl:.4f}  '
              f'train_acc={ta:.1f}%  val_acc={va:.1f}%')

In [ ]:
plot_history(tl_ce, vl_ce, ta_ce, va_ce, 'Baseline (no DA) — Training Curves')
CE_model.load_state_dict(best_state_ce)
res_ce_s_t, res_ce_s_p, res_ce_s_prob, res_ce_t_t, res_ce_t_p, res_ce_t_prob = \
    full_eval(CE_model, source_test_loader, target_test_loader, device, 'Baseline (no DA)')

### Reflection

Look at the source and target accuracy numbers, the confusion matrices, and the t-SNE plot.

- How large is the accuracy gap? Does it match your prediction?
- The domain shift here is purely visual (textured backgrounds). Why would a CNN trained on clean digits fail on textured ones, even though the digit shapes are identical?
- Look at the per-class accuracy in the classification report. Are all three digits affected equally by the domain shift, or do some transfer better than others?
- In the t-SNE, are source and target points for the same class clustered together, or separated? What does this suggest about how the CNN is using texture vs. shape features?

## 5. MMD Domain Adaptation

<p align="center">
<img img src="https://github.com/AleksCipri/LSST_DSFP_DA/raw/main/mmd.png" alt="fine" width="400"/>
</p>

We add a **Maximum Mean Discrepancy** (MMD) penalty that encourages the source and target latent distributions to overlap:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CE}} + \lambda_{\text{MMD}} \cdot \mathcal{L}_{\text{MMD}}$$

The MMD² between two distributions $\mathbb{P}_s$ and $\mathbb{P}_t$ with RBF kernel $k(z, z') = \exp\!\left(-\frac{\|z-z'\|^2}{2\sigma^2}\right)$ is:

$$\mathcal{L}_{\text{MMD}} = \mathbb{E}_{z,z' \sim \mathbb{P}_s}[k(z,z')] + \mathbb{E}_{\tilde{z},\tilde{z}' \sim \mathbb{P}_t}[k(\tilde{z},\tilde{z}')] - 2\,\mathbb{E}_{z \sim \mathbb{P}_s,\,\tilde{z} \sim \mathbb{P}_t}[k(z,\tilde{z})]$$

We use a mixture of RBF kernels with bandwidths $\sigma \in \{1, 5, 10, 50\}$ to capture structure at multiple scales. **Target labels are never used.**

**Questions to consider:**
- What does MMD² = 0 mean geometrically in the latent space?
- We compute MMD on 64-dimensional latent vectors $z$, not on 28×28×3 pixel tensors. Why is this a better choice?
- What happens if $\lambda_{\text{MMD}}$ is too large? What if it is too small?

In [ ]:
def _rbf_kernel(x: torch.Tensor, y: torch.Tensor, sigma: float) -> torch.Tensor:
    x_sq = x.pow(2).sum(1, keepdim=True)
    y_sq = y.pow(2).sum(1, keepdim=True)
    # clamp prevents small negative values from floating-point cancellation
    # amplifying via exp with small sigma, which would produce NaN losses
    dist = torch.clamp(x_sq + y_sq.t() - 2.0 * x @ y.t(), min=0.0)
    return torch.exp(-dist / (2.0 * sigma ** 2))


class MMDLoss(nn.Module):
    """
    Unbiased MMD² with a mixture of RBF kernels.
    sigmas : kernel bandwidths — multiple scales capture structure at different resolutions.
    """
    def __init__(self, sigmas: Optional[List[float]] = None):
        super().__init__()
        self.sigmas = sigmas or [1.0, 5.0, 10.0, 50.0]

    def forward(self, source: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ## TO DO ##
        # For each sigma in self.sigmas, compute k_ss, k_tt, k_st using _rbf_kernel,
        # take the mean of each, and accumulate: mmd += k_ss.mean() + k_tt.mean() - 2*k_st.mean()
        # Initialise mmd = torch.tensor(0.0, device=source.device) before the loop.
        # Return mmd.
        pass

### MMD Training Loop

The key difference from the baseline: **both source and target batches are passed through the model together** in each step. We use the latent vectors to compute the MMD penalty, but only the source logits for the classification loss.

**Your tasks** (marked with `## TO DO ##`):
1. Concatenate source and target image batches along the batch dimension.
2. Run the combined batch through the model to get `logits` and `z`.
3. Split `logits` and `z` back into source and target portions.
4. Compute `ce_loss` (source only) and `mmd_loss` (source vs. target latents), then combine.

In [ ]:
MMD_model   = CNN(num_classes=3).to(device)
criterion   = nn.CrossEntropyLoss()
optimizer   = optim.Adam(MMD_model.parameters(), lr=1e-3)
mmd_loss_fn = MMDLoss()
lambda_mmd  = 0.2

tl_mmd, vl_mmd, ta_mmd, va_mmd, mmd_losses = [], [], [], [], []
best_val_m, best_state_m = float('inf'), None

for epoch in tqdm(range(NUM_EPOCHS)):
    MMD_model.train()
    t_ce = t_da = t_correct = t_total = 0
    tgt_iter = iter(target_train_loader)

    for imgs_s, labels_s in source_train_loader:
        try:    imgs_t, _ = next(tgt_iter)
        except: tgt_iter = iter(target_train_loader); imgs_t, _ = next(tgt_iter)
        imgs_s, labels_s = imgs_s.to(device), labels_s.to(device)
        imgs_t = imgs_t.to(device)

        ## TO DO ##
        # 1. Concatenate imgs_s and imgs_t into a single tensor `combined`, so both domains
        #    pass through the model in one forward pass.
        #
        # 2. Run the combined batch through MMD_model to obtain class logits and
        #    the latent feature vector z for every image.
        #
        # 3. Split z and logits back into their source and target portions using
        #    the known source batch size as the boundary index.
        #
        # 4. Compute the classification loss on source logits and labels.
        #    Compute the MMD loss between source and target latent features.
        #    Combine them into a single scalar using lambda_mmd as the trade-off weight.

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        t_ce      += ce_loss.item()
        t_da      += mmd_loss.item()
        t_correct += logits_s.argmax(1).eq(labels_s).sum().item()
        t_total   += labels_s.size(0)

    MMD_model.eval()
    v_loss = v_correct = v_total = 0
    with torch.no_grad():
        for imgs, labels in source_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _  = MMD_model(imgs)
            v_loss    += criterion(logits, labels).item()
            v_correct += logits.argmax(1).eq(labels).sum().item()
            v_total   += labels.size(0)

    n   = len(source_train_loader)
    tl  = t_ce / n                          # train CE loss
    vl  = v_loss / len(source_val_loader)   # val CE loss
    ta  = 100 * t_correct / t_total
    va  = 100 * v_correct / v_total
    mmd = t_da / n                          # MMD loss (logged separately)

    tl_mmd.append(tl); vl_mmd.append(vl)
    ta_mmd.append(ta); va_mmd.append(va)
    mmd_losses.append(mmd)

    if vl < best_val_m:
        best_val_m = vl; best_state_m = copy.deepcopy(MMD_model.state_dict())
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}  CE={tl:.4f}  MMD={mmd:.4f}  '
              f'val_CE={vl:.4f}  train_acc={ta:.1f}%  val_acc={va:.1f}%')

In [ ]:
plot_history(tl_mmd, vl_mmd, ta_mmd, va_mmd, 'MMD Domain Adaptation — Training Curves')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(mmd_losses)+1), mmd_losses, color='tab:orange')
ax.set_xlabel('Epoch'); ax.set_ylabel('MMD\u00b2 Loss')
ax.set_title('MMD Domain Adaptation — RBF Mixture MMD\u00b2 per Epoch')
plt.tight_layout(); plt.show()

MMD_model.load_state_dict(best_state_m)
res_mmd_s_t, res_mmd_s_p, res_mmd_s_prob, res_mmd_t_t, res_mmd_t_p, res_mmd_t_prob = \
    full_eval(MMD_model, source_test_loader, target_test_loader, device, 'MMD Domain Adaptation')

### Reflection

- Look at the MMD² curve. Is it decreasing over epochs? What does a decreasing MMD² mean for the alignment of source and target latent distributions?
- Compare the t-SNE before and after MMD. Do source (circles) and target (triangles) points of the same class overlap more?
- Did target accuracy improve compared to the baseline? Did source accuracy stay stable, or did it drop? If it dropped, what might explain that trade-off?
- **Potential pitfall:** if you set `lambda_mmd` very large (e.g. 10), what do you expect would happen to the CE loss and classification accuracy? Try it and record the result.

## 6. Domain-Adversarial Neural Network (DANN)

<p align="center">
<img img src="https://adapt-python.github.io/adapt/_images/dann.png" alt="fine" width="600"/>
</p>

In DANN, a **domain classifier** $G_d$ tries to distinguish MNIST from MNIST-M latents. A **gradient reversal layer (GRL)** flips the sign of gradients flowing *back* through it, so the feature extractor learns to *fool* the domain classifier — forcing it to produce domain-invariant representations.

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{CE}} + \lambda \cdot \mathcal{L}_{\text{domain}}$$

The domain classifier minimises $\mathcal{L}_{\text{domain}}$ (binary cross-entropy: source=1, target=0). The feature extractor *maximises* it via the GRL.

**Questions to consider before training:**
- At the adversarial optimum, the domain classifier is at chance (50% accuracy). What value does $\mathcal{L}_{\text{domain}}$ converge to for binary cross-entropy at 50% accuracy? *(Hint: $-\log(0.5)$)*
- Should the domain loss **increase** or **decrease** during training as DA improves?
- What could go wrong if $\lambda$ is too large early in training, before the feature extractor has learned anything useful?
- The GRL is an identity function in the forward pass and negates gradients in the backward pass. Why is this trick necessary — couldn't we just multiply the domain loss by $-\lambda$ directly?

In [ ]:
class GradientReversal(Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)   # identity in the forward pass

    @staticmethod
    def backward(ctx, grad):
        ## TO DO ##
        # Return the negated, scaled gradient for x, and None for lam.
        # The gradient reversal is: -lam * grad.
        # This single line is the entire mechanism that makes DANN adversarial.
        pass

def grad_reverse(x, lam=1.0): # the scaling of the DA loss is basically done using lam
    return GradientReversal.apply(x, lam)


class DomainClassifier(nn.Module):
    def __init__(self, z_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 32), nn.ReLU(),
            nn.Linear(32, 1),   # single logit — used with BCEWithLogitsLoss
        )
    def forward(self, z, lam=1.0):
        return self.net(grad_reverse(z, lam))

### DANN Training Loop

**Your tasks** (marked with `## TO DO ##`):
1. Build `dom_labels`: a column vector of ones for source images and zeros for target images.
2. Concatenate source and target batches and run through `DANN_model`.
3. Split `logits` and `z` into source and target portions.
4. Compute `loss_ce` (source classification) and `loss_dom` (domain classification via `domain_clf`), then combine.

In [ ]:
DANN_model  = CNN(num_classes=3).to(device)
domain_clf  = DomainClassifier(z_dim=64).to(device)
optimizer_d = optim.Adam(
    list(DANN_model.parameters()) + list(domain_clf.parameters()), lr=1e-3)
criterion_ce  = nn.CrossEntropyLoss()
criterion_dom = nn.BCEWithLogitsLoss()
lambda_grl    = 0.2

tl_d, vl_d, ta_d, va_d, dom_losses = [], [], [], [], []
best_val_d, best_state_d = float('inf'), None

for epoch in tqdm(range(NUM_EPOCHS)):
    DANN_model.train(); domain_clf.train()
    t_ce = t_da = t_correct = t_total = 0
    tgt_iter = iter(target_train_loader)

    for imgs_s, labels_s in source_train_loader:
        try:    imgs_t, _ = next(tgt_iter)
        except: tgt_iter = iter(target_train_loader); imgs_t, _ = next(tgt_iter)
        imgs_s, labels_s = imgs_s.to(device), labels_s.to(device)
        imgs_t = imgs_t.to(device)

        ## TO DO ##
        # 1. Create domain labels: a float tensor of 1s for source images and 0s for target
        #    images, shaped (batch_size, 1) each, then concatenate them into a single tensor
        #    on the correct device.
        #
        # 2. Concatenate source and target images along the batch dimension, then pass the
        #    combined batch through DANN_model to get logits and latent features z.
        #
        # 3. Slice out only the source portion of logits (first imgs_s.size(0) rows)
        #    for the classification loss.
        #
        # 4. Compute the classification loss on source logits vs source labels.
        #    Compute the domain loss by passing z through the domain classifier (with
        #    gradient reversal applied via lambda_grl) against the domain labels.
        #    Sum both losses into a single scalar representing the total loss.


        optimizer_d.zero_grad(); loss.backward(); optimizer_d.step()
        t_ce     += loss_ce.item()
        t_da     += loss_dom.item()
        t_correct += logits_s.argmax(1).eq(labels_s).sum().item()
        t_total   += labels_s.size(0)

    DANN_model.eval()
    v_loss = v_correct = v_total = 0
    with torch.no_grad():
        for imgs, labels in source_val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _  = DANN_model(imgs)
            v_loss    += criterion_ce(logits, labels).item()
            v_correct += logits.argmax(1).eq(labels).sum().item()
            v_total   += labels.size(0)

    n   = len(source_train_loader)
    tl  = t_ce / n                          # train CE loss
    vl  = v_loss / len(source_val_loader)   # val CE loss
    ta  = 100 * t_correct / t_total
    va  = 100 * v_correct / v_total
    dom = t_da / n                          # domain loss (logged separately)

    tl_d.append(tl); vl_d.append(vl)
    ta_d.append(ta); va_d.append(va)
    dom_losses.append(dom)

    if vl < best_val_d:
        best_val_d = vl; best_state_d = copy.deepcopy(DANN_model.state_dict())
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}  CE={tl:.4f}  DOM={dom:.4f}  '
              f'val_CE={vl:.4f}  train_acc={ta:.1f}%  val_acc={va:.1f}%')

In [ ]:
plot_history(tl_d, vl_d, ta_d, va_d, 'DANN — Training Curves')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(dom_losses)+1), dom_losses, color='tab:red')
ax.axhline(y=0.693, color='gray', linestyle='--', lw=1,
           label='ln(2) \u2248 0.693 \u2014 adversarial optimum')
ax.legend(loc='upper right')
ax.set_xlabel('Epoch'); ax.set_ylabel('Domain Loss')
ax.set_title('DANN — Domain Classifier Loss per Epoch')
ax.legend(); plt.tight_layout(); plt.show()

DANN_model.load_state_dict(best_state_d)
res_d_s_t, res_d_s_p, res_d_s_prob, res_d_t_t, res_d_t_p, res_d_t_prob = \
    full_eval(DANN_model, source_test_loader, target_test_loader, device, 'DANN')

### Reflection

- The dashed line marks $\ln(2) \approx 0.693$ — the binary cross-entropy at 50% accuracy (random guessing). Is the domain loss approaching this value? If it stays well below, what does that mean for the adversarial game?
- Compare the DANN t-SNE to the MMD t-SNE. Which method produced better-aligned latent clusters for this digit classification task?
- DANN requires tuning two things: `lambda_grl` and the domain classifier capacity. Try making the domain classifier deeper (add a hidden layer). Does a stronger adversary help or hurt adaptation?
- The GRL is conceptually elegant but has known instability issues. What did you observe in the domain loss curve — is it smooth or jagged?

## 7. Common Training Difficulties and Pitfalls

Both MMD and DANN can fail silently — the training loss decreases but target accuracy does not improve. Here are the most common failure modes:

**MMD**
- **$\lambda_{\text{MMD}}$ too large**: the model prioritises alignment over classification. The CE loss stops decreasing and source accuracy collapses. The MMD² goes to zero quickly but the classifier is useless.
- **$\lambda_{\text{MMD}}$ too small**: alignment is negligible. The model behaves like the baseline.
- **Kernel bandwidth mismatch**: if all $\sigma$ values are much smaller than the typical pairwise distances in the latent space, all kernel values are near zero and the gradient signal vanishes. The clamping in `_rbf_kernel` guards against NaN but not against zero-gradient.
- **Negative transfer**: forcing alignment can hurt if the source and target class distributions are skewed relative to each other (e.g. one domain has more of digit 4 than the other).

**DANN**
- **Domain classifier too strong early**: if the domain classifier converges faster than the feature extractor, it provides a vanishing gradient signal — the GRL reversal of a near-zero gradient does nothing.
- **$\lambda_{\text{GRL}}$ too large**: catastrophic forgetting — the feature extractor destroys classification-relevant features in its rush to fool the domain classifier.
- **Domain loss does not rise toward $\ln(2)$**: the adversarial game has not reached equilibrium. Either the domain classifier is winning (loss stays low) or the feature extractor collapsed (both losses are noisy).
- **Gradient instability**: because the GRL flips signs, loss curves can be jagged. Gradient clipping (`torch.nn.utils.clip_grad_norm_`) can help stabilise training.

**General**
- **Forgetting to use the validation set for model selection**: saving the best checkpoint by *training* loss can overfit; always select on source validation CE loss.
- **Normalising source and target together**: leaks target statistics into training. Always compute normalisation statistics separately per domain.
- **Class imbalance across domains**: if the class ratio differs between MNIST and MNIST-M subsets, MMD and DANN may align the wrong distributions. Check that each digit is roughly equally represented in both train splits.

## 8. Model Comparison

Side-by-side macro-averaged ROC curves and a summary accuracy / AUC table for all three models on both domains.

In [ ]:
def macro_roc(y_true, y_prob):
    """Interpolated macro-average ROC across all classes (OvR)."""
    y_bin    = label_binarize(y_true, classes=list(CLASS_NAMES.keys()))
    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    for i in range(len(CLASS_NAMES)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        tprs.append(np.interp(mean_fpr, fpr, tpr))
    macro_tpr = np.mean(tprs, axis=0)
    return mean_fpr, macro_tpr, float(auc(mean_fpr, macro_tpr))

models_info = [
    ('Baseline', CE_model,   res_ce_s_t,  res_ce_s_prob,  res_ce_t_t,  res_ce_t_prob),
    ('MMD-DA',   MMD_model,  res_mmd_s_t, res_mmd_s_prob, res_mmd_t_t, res_mmd_t_prob),
    ('DANN',     DANN_model, res_d_s_t,   res_d_s_prob,   res_d_t_t,   res_d_t_prob),
]

# Combined macro ROC curves (interpolates the per-class OvR curves onto a shared FPR grid and averages them)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Macro-Averaged ROC Curves — All Models', fontsize=13)
for name, _, ys_t, ys_prob, yt_t, yt_prob in models_info:
    fpr_s, tpr_s, auc_s = macro_roc(ys_t, ys_prob)
    ax1.plot(fpr_s, tpr_s, lw=2, label=f'{name}  (AUC={auc_s:.3f})')
    fpr_t, tpr_t, auc_t = macro_roc(yt_t, yt_prob)
    ax2.plot(fpr_t, tpr_t, lw=2, label=f'{name}  (AUC={auc_t:.3f})')
for ax, title in [(ax1, 'Source (MNIST)'), (ax2, 'Target (MNIST-M)')]:
    ax.plot([0,1],[0,1], 'k--', lw=0.8)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

# Summary table
print(f'\n{"Model":<20} {"Src Acc":>10} {"Tgt Acc":>10} {"Src AUC":>10} {"Tgt AUC":>10}')
print('-' * 60)
for name, model, ys_t, ys_prob, yt_t, yt_prob in models_info:
    ys_p = get_predictions(model, source_test_loader, device)[1]
    yt_p = get_predictions(model, target_test_loader, device)[1]
    src_acc = 100*(ys_p == ys_t).mean()
    tgt_acc = 100*(yt_p == yt_t).mean()
    src_auc = roc_auc_score(ys_t, ys_prob, multi_class='ovr', average='macro')
    tgt_auc = roc_auc_score(yt_t, yt_prob, multi_class='ovr', average='macro')
    print(f'{name:<20} {src_acc:>9.2f}% {tgt_acc:>9.2f}% {src_auc:>10.3f} {tgt_auc:>10.3f}')

### Final Questions

- Which method closed the most of the source–target gap? Were you surprised by the result?
- Look at the source accuracy across all three models. Did applying DA hurt source performance? Is a small source-accuracy drop acceptable if target accuracy improves?
- The t-SNE shows the latent space *after* training. What would you want to see in an ideal DA result — and how close did you get?
- Digit 1 is visually very distinct from 4 and 8. Did this digit transfer better than the others? Check per-class accuracy in the classification reports.
- Now that you have completed both the MNIST and Mergers tutorials: which domain shift is harder to close — texture-based (MNIST → MNIST-M) or noise-based (pristine → noisy galaxies)? What properties of each shift explain the difficulty?

## 9. Extensions

Here are some directions to explore on your own:

**Data**
- Train on all 10 MNIST digits instead of just 1, 4, 8. How does the domain gap change? How do the t-SNE plots look with 10 classes?
- Swap the domains: train on MNIST-M and evaluate on MNIST. Does adaptation help equally in both directions, or is one direction harder?
- Compute **per-class accuracy** on the target domain: which digits transfer best and which suffer most from the shift?

**Model architecture**
- Try a deeper CNN or a pre-trained backbone (e.g. a small ResNet fine-tuned from ImageNet weights). How does a more expressive feature extractor affect domain alignment?
- Make the domain classifier in DANN deeper or shallower. Does a stronger adversary help or hurt?

**Training**
- Add a learning rate scheduler (`CosineAnnealingLR` or `ReduceLROnPlateau`) and observe its effect on convergence.
- Test increasing or decreasing DA loss weight $\lambda$. What happens when you do this? Can the optimal weight be learned?
- For DANN, gradually increase $\lambda$ during training using the schedule from the original paper: $\lambda = \frac{2}{1+e^{-10p}} - 1$, where $p$ is the fraction of training completed. This stabilises early training when the feature extractor is still random.

**MMD**
- Experiment with different kernel bandwidth sets in `MMDLoss`. What happens with only one $\sigma$ vs. many? Can you find a better set for this dataset?
- Implement a **median heuristic** to set $\sigma$ automatically from the pairwise distances in each mini-batch.
- Try using the [**GeomLoss**](https://www.kernel-operations.io/geomloss/) package and compare Sinkhorn (Wasserstein) loss to MMD.

**Evaluation**
- Monitor target validation accuracy *during* training (without using it for model selection) to see how domain alignment progresses epoch by epoch.
- Try [**UMAP**](https://umap-learn.readthedocs.io/en/latest/) as an alternative to t-SNE — it is faster and often preserves global cluster structure better for high-dimensional latent spaces.
- Compute **per-class accuracy** on the target domain: which digits transfer best and which suffer most from the domain shift?

**Other DA methods**
- Implement [**CORAL**](https://arxiv.org/pdf/1612.01939) (Correlation Alignment): minimise the Frobenius norm between source and target second-order feature statistics — much simpler than MMD or DANN and often surprisingly competitive.

Happy coding!